# ML Crystal Classifier

This notebook is a cleaned version of an IOAI-style feature-engineering challenge.

The task is to classify signal-like `187 × 8` arrays into two classes. The model is intentionally constrained to a depth-limited Decision Tree, so the main work is designing useful statistical features from the raw signal arrays.

## Approach

1. Load `187 × 8` signal arrays and binary labels.
2. Normalize each sample independently so the classifier focuses more on signal shape than absolute scale.
3. Extract compact statistical features from each channel.
4. Train a depth-limited `DecisionTreeClassifier`.
5. Report validation accuracy and ROC-AUC.

The strongest feature set in the original exploratory notebook reached about **93.08% validation score** in the provided validation setup.

In [ ]:
import sys
from pathlib import Path

# Allow notebook to import from ../src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.features import extract_statistical_features, extract_hjorth_features
from src.evaluate import evaluate_classifier
from src.visualize import plot_signal_sample

## Data loading

This repository does not include the original challenge dataset. Place `FE_hometask_data_v3.pickle` in the project root, or use the original challenge download command if you have access to it.

Expected data structure:

```python
data["train"] = {id: (array_187x8, label), ...}
data["val"] = {id: (array_187x8, label), ...}
```

In [ ]:
DATA_PATH = PROJECT_ROOT / "FE_hometask_data_v3.pickle"

if not DATA_PATH.exists():
    print(f"Dataset not found at {DATA_PATH}.")
    print("Place FE_hometask_data_v3.pickle in the project root before running the full notebook.")
else:
    data = pd.read_pickle(DATA_PATH)

    X_train = np.array([x[0] for x in data["train"].values()])
    y_train = np.array([x[1] for x in data["train"].values()])

    X_val = np.array([x[0] for x in data["val"].values()])
    y_val = np.array([x[1] for x in data["val"].values()])

    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)
    print("X_val:", X_val.shape)
    print("y_val:", y_val.shape)

## Visualize examples

In [ ]:
if DATA_PATH.exists():
    plot_signal_sample(X_train[0], label=f"class={y_train[0]}")
    plot_signal_sample(X_train[4], label=f"class={y_train[4]}")

## Baseline features

A simple baseline is to use per-channel standard deviation over the time axis.

In [ ]:
def baseline_std_features(X):
    return np.std(X, axis=1)

if DATA_PATH.exists():
    baseline_train = baseline_std_features(X_train)
    baseline_val = baseline_std_features(X_val)

    baseline_scores = evaluate_classifier(
        baseline_train,
        baseline_val,
        y_train,
        y_val,
        max_depth=20,
    )

    baseline_scores

## Final statistical feature set

The final feature extractor normalizes each sample and combines statistics like standard deviation, finite differences, skewness, max values, kurtosis, and higher-order moments.

In [ ]:
if DATA_PATH.exists():
    train_features = extract_statistical_features(X_train)
    val_features = extract_statistical_features(X_val)

    print("Feature matrix:", train_features.shape)

    final_scores = evaluate_classifier(
        train_features,
        val_features,
        y_train,
        y_val,
        max_depth=20,
    )

    final_scores

## Alternative exploratory feature set

This uses signal-inspired Hjorth-style features: activity, mobility, and complexity.

In [ ]:
if DATA_PATH.exists():
    hjorth_train = extract_hjorth_features(X_train)
    hjorth_val = extract_hjorth_features(X_val)

    hjorth_scores = evaluate_classifier(
        hjorth_train,
        hjorth_val,
        y_train,
        y_val,
        max_depth=20,
    )

    hjorth_scores

## Report

In this project, I treated each `187 × 8` array as a multi-channel signal and focused on feature engineering rather than using a larger model. I normalized each sample independently, then extracted statistical descriptors capturing spread, asymmetry, sharpness, high-order distribution shape, and changes across channels. These engineered features allowed a depth-limited Decision Tree to separate the two classes effectively, reaching about **93.08% validation score** in the original challenge setup.